# Sustainable AI in Healthcare - Quick Start Guide

This notebook demonstrates how to use the Sustainable AI in Healthcare pipeline with a sample dataset.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification, make_regression
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# Import our pipeline
import sys
sys.path.append('../src')
from pipeline import SustainableAIPipeline

## 1. Create Sample Healthcare Data

Let's create a synthetic healthcare dataset for demonstration.


In [ ]:
# Create synthetic healthcare data for classification
n_samples = 1000
n_features = 15

# Generate base features
X, y = make_classification(
    n_samples=n_samples,
    n_features=n_features,
    n_informative=8,
    n_redundant=3,
    n_classes=2,
    class_sep=1.2,
    random_state=42
)

# Create meaningful feature names for healthcare context
feature_names = [
    'age', 'bmi', 'blood_pressure_systolic', 'blood_pressure_diastolic',
    'cholesterol_total', 'cholesterol_ldl', 'cholesterol_hdl', 'glucose_fasting',
    'heart_rate', 'temperature', 'respiratory_rate', 'oxygen_saturation',
    'hemoglobin', 'white_blood_cells', 'platelets'
]

# Create DataFrame
df = pd.DataFrame(X, columns=feature_names)

# Add some realistic healthcare data transformations
df['age'] = np.clip(df['age'] * 20 + 50, 18, 90).astype(int)
df['bmi'] = np.clip(df['bmi'] * 5 + 25, 15, 45)
df['blood_pressure_systolic'] = np.clip(df['blood_pressure_systolic'] * 20 + 120, 80, 200)
df['blood_pressure_diastolic'] = np.clip(df['blood_pressure_diastolic'] * 15 + 80, 50, 120)
df['heart_rate'] = np.clip(df['heart_rate'] * 20 + 70, 40, 150)

# Add target variable (e.g., risk of cardiovascular disease)
df['cvd_risk'] = y

# Add some categorical features
df['gender'] = np.random.choice(['Male', 'Female'], n_samples)
df['smoking_status'] = np.random.choice(['Never', 'Former', 'Current'], n_samples, p=[0.5, 0.3, 0.2])
df['diabetes'] = np.random.choice(['No', 'Yes'], n_samples, p=[0.8, 0.2])

# Add some missing values to make it realistic
missing_cols = ['cholesterol_ldl', 'glucose_fasting', 'hemoglobin']
for col in missing_cols:
    missing_idx = np.random.choice(df.index, int(0.1 * len(df)), replace=False)
    df.loc[missing_idx, col] = np.nan

print(f"Dataset shape: {df.shape}")
print(f"Target distribution: {df['cvd_risk'].value_counts()}")
print(f"Missing values: {df.isnull().sum().sum()}")

# Save the dataset
df.to_csv('../data/sample_healthcare_data.csv', index=False)
print("Sample dataset saved to '../data/sample_healthcare_data.csv'")

In [ ]:
# Display first few rows
df.head()

## 2. Initialize the Sustainable AI Pipeline

Let's initialize our pipeline with the default configuration.


In [ ]:
# Initialize the pipeline
pipeline = SustainableAIPipeline(config_path="../configs/default_config.yaml")

print("Pipeline initialized successfully!")
print(f"Configuration loaded: {len(pipeline.config)} main sections")

## 3. Run the Complete Pipeline

Now let's run the complete 8-step pipeline on our healthcare data.


In [ ]:
# Run the complete pipeline
try:
    results = pipeline.run_full_pipeline('../data/sample_healthcare_data.csv')
    print("Pipeline completed successfully!")
    
    # Display pipeline status
    status = pipeline.get_pipeline_status()
    print("\nPipeline Status:")
    for step, completed in status.items():
        status_symbol = "✅" if completed else "❌"
        print(f"{status_symbol} {step.replace('_', ' ').title()}")
        
except Exception as e:
    print(f"Pipeline failed: {e}")
    import traceback
    traceback.print_exc()

## 4. Analyze Results

Let's examine the results from each step of the pipeline.


In [ ]:
# Analyze data collection results
if 'data_collection' in results:
    dc_results = results['data_collection']
    print("📊 Data Collection Results:")
    print(f"  - Data shape: {dc_results.get('data_shape', 'N/A')}")
    
    validation = dc_results.get('validation_report', {})
    if validation:
        print(f"  - Total records: {validation.get('total_records', 'N/A')}")
        print(f"  - Total features: {validation.get('total_features', 'N/A')}")
        print(f"  - Issues found: {len(validation.get('issues_found', []))}")
    print()

In [ ]:
# Analyze preprocessing results
if 'preprocessing' in results:
    prep_results = results['preprocessing']
    print("🔧 Data Preprocessing Results:")
    print(f"  - Original shape: {prep_results.get('original_shape', 'N/A')}")
    print(f"  - Processed shape: {prep_results.get('processed_shape', 'N/A')}")
    
    report = prep_results.get('preprocessing_report', {})
    if report:
        print(f"  - Steps applied: {len(report.get('steps_applied', []))}")
        shape_change = report.get('shape_change', {})
        print(f"  - Rows removed: {shape_change.get('rows_removed', 0)}")
        print(f"  - Columns added: {shape_change.get('columns_added', 0)}")
    print()

In [ ]:
# Analyze model development results
if 'model_development' in results:
    model_results = results['model_development']
    print("🤖 Model Development Results:")
    training_report = model_results.get('training_report', {})
    
    if training_report:
        print(f"  - Task type: {training_report.get('task_type', 'N/A')}")
        print(f"  - Best model: {training_report.get('best_model', 'N/A')}")
        
        models_trained = training_report.get('models_trained', {})
        print(f"  - Models trained: {len(models_trained)}")
        
        for model_name, model_info in models_trained.items():
            if model_info.get('status') == 'success':
                cv_score = model_info.get('cross_val_score', {})
                if isinstance(cv_score, dict):
                    score = cv_score.get('mean', 'N/A')
                else:
                    score = cv_score
                print(f"    - {model_name}: {score:.4f}" if isinstance(score, float) else f"    - {model_name}: {score}")
    print()

In [ ]:
# Analyze evaluation results
if 'evaluation' in results:
    eval_results = results['evaluation']
    print("📈 Model Evaluation Results:")
    
    eval_report = eval_results.get('evaluation_report', {})
    if eval_report:
        test_info = eval_report.get('test_set_info', {})
        print(f"  - Test samples: {test_info.get('samples', 'N/A')}")
        print(f"  - Test features: {test_info.get('features', 'N/A')}")
        
        model_comparison = eval_report.get('model_comparison', {})
        if model_comparison:
            print(f"  - Best model: {model_comparison.get('best_model', 'N/A')}")
            
            ranking = model_comparison.get('ranking', [])
            if ranking:
                print("  - Model ranking:")
                for i, model_rank in enumerate(ranking[:3]):
                    print(f"    {i+1}. {model_rank.get('model', 'N/A')}: {model_rank.get('score', 0):.4f}")
    print()

## 5. Step-by-Step Pipeline Execution

Let's also demonstrate how to run the pipeline step by step for more control.


In [ ]:
# Initialize a new pipeline for step-by-step execution
step_pipeline = SustainableAIPipeline()

# Step 1: Data Collection
print("Step 1: Data Collection and Understanding")
data_results = step_pipeline.collect_and_understand_data('../data/sample_healthcare_data.csv')
print(f"✅ Data collected: {data_results['data_shape']}")

# Step 2: Preprocessing
print("\nStep 2: Data Preprocessing")
prep_results = step_pipeline.preprocess_data()
print(f"✅ Data preprocessed: {prep_results['original_shape']} → {prep_results['processed_shape']}")

# Step 3: EDA
print("\nStep 3: Exploratory Data Analysis")
eda_results = step_pipeline.perform_eda()
print("✅ EDA completed")

# Step 4: Feature Engineering
print("\nStep 4: Feature Engineering")
fe_results = step_pipeline.engineer_features()
print(f"✅ Features engineered: {fe_results['original_features']} → {fe_results['final_features']}")
print(f"   Task type detected: {fe_results['task_type']}")

In [ ]:
# Continue with remaining steps
# Step 5: Model Development
print("Step 5: Model Development")
model_results = step_pipeline.develop_models()
print(f"✅ Models trained: {', '.join(model_results['models_trained'])}")

# Step 6: Evaluation
print("\nStep 6: Model Evaluation")
eval_results = step_pipeline.evaluate_models()
print("✅ Models evaluated")

# Step 7: Federated Learning
print("\nStep 7: Federated Learning Setup")
fl_results = step_pipeline.implement_federated_learning()
print(f"✅ {fl_results['message']}")

# Step 8: Interpretation
print("\nStep 8: Model Interpretation and Insights")
interp_results = step_pipeline.interpret_and_discuss()
print("✅ Model interpretation completed")

## 6. View Generated Reports and Visualizations

The pipeline generates various reports and visualizations. Let's check what was created.


In [ ]:
import os
from pathlib import Path

# Check generated files
reports_dir = Path("../reports")
if reports_dir.exists():
    print("📁 Generated Reports and Files:")
    
    for root, dirs, files in os.walk(reports_dir):
        level = root.replace(str(reports_dir), '').count(os.sep)
        indent = ' ' * 2 * level
        print(f"{indent}{os.path.basename(root)}/")
        
        sub_indent = ' ' * 2 * (level + 1)
        for file in files:
            print(f"{sub_indent}{file}")
else:
    print("No reports directory found yet.")

In [ ]:
# Check models directory
models_dir = Path("../models")
if models_dir.exists():
    print("🤖 Generated Model Files:")
    
    for file in models_dir.glob("*"):
        if file.is_file():
            size_mb = file.stat().st_size / (1024 * 1024)
            print(f"  - {file.name} ({size_mb:.2f} MB)")
        elif file.is_dir():
            print(f"  - {file.name}/ (directory)")
else:
    print("No models directory found yet.")

## 7. Sustainability Analysis

Let's examine the sustainability metrics of our pipeline.


In [ ]:
# Check if interpretation results include sustainability analysis
if hasattr(step_pipeline, 'interpretation_results'):
    interp_results = step_pipeline.interpretation_results
    
    sustainability = interp_results.get('sustainability_analysis', {})
    if sustainability:
        print("🌱 Sustainability Analysis:")
        
        # Energy metrics
        energy_metrics = sustainability.get('energy_metrics', {})
        if energy_metrics:
            print(f"  Energy Consumption:")
            print(f"    - Total: {energy_metrics.get('total_energy_kwh', 0):.4f} kWh")
            print(f"    - Training: {energy_metrics.get('training_energy_kwh', 0):.4f} kWh")
        
        # Carbon footprint
        carbon_footprint = sustainability.get('carbon_footprint', {})
        if carbon_footprint:
            print(f"  Carbon Footprint:")
            print(f"    - CO₂ Emissions: {carbon_footprint.get('co2_emissions_kg', 0):.4f} kg")
            print(f"    - Tree-months equivalent: {carbon_footprint.get('equivalent_tree_months', 0):.2f}")
        
        # Sustainability score
        score = sustainability.get('sustainability_score', 0)
        print(f"  Overall Sustainability Score: {score:.1f}/100")
        
        # Recommendations
        recommendations = sustainability.get('recommendations', [])
        if recommendations:
            print(f"  Recommendations:")
            for rec in recommendations[:3]:
                print(f"    - {rec}")
    else:
        print("Sustainability analysis not available.")
else:
    print("Interpretation results not available.")

## 8. Summary and Next Steps

Congratulations! You've successfully run the complete Sustainable AI in Healthcare pipeline.


In [ ]:
print("🎉 Pipeline Execution Complete!")
print("\n📋 Summary:")
print("  ✅ Data collected and validated")
print("  ✅ Data preprocessed and cleaned")
print("  ✅ Exploratory data analysis performed")
print("  ✅ Features engineered and selected")
print("  ✅ Multiple models trained and optimized")
print("  ✅ Models evaluated and compared")
print("  ✅ Federated learning framework initialized")
print("  ✅ Models interpreted with explainability analysis")
print("  ✅ Sustainability metrics calculated")

print("\n🚀 Next Steps:")
print("  1. Explore generated reports and visualizations")
print("  2. Fine-tune models based on evaluation results")
print("  3. Deploy best model with monitoring")
print("  4. Set up federated learning with real clients")
print("  5. Implement continuous model monitoring")

print("\n📚 Resources:")
print("  - Configuration: configs/default_config.yaml")
print("  - Documentation: README.md")
print("  - Models: models/ directory")
print("  - Reports: reports/ directory")